# CNN + Embeddings from ESM-2


## 1. Setup and Data Loading
Imports libraries and loads the protein dataset for K-fold cross-validation.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv')

if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

print(df.head())
df.info()

## 2. Load Pre-trained ESM-2 Model
This is used to generate the frozen embeddings.

In [ ]:
# Load ESM-2 model and alphabet
esm_model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t30_150M_UR50D")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# Get the maximum sequence length from the dataframe
max_len = df["len"].max()

## 3. Generate Frozen Embeddings


In [ ]:
# Prepare data for batching
sequences = [s.replace("*", "X") for s in df['seq'].tolist()]
labels = df['pdb_id'].tolist()
data = list(zip(labels, sequences))

batch_size = 8
all_embeddings = []

for i in tqdm(range(0, len(data), batch_size), desc="Generating Embeddings"):
    batch_data = data[i:i+batch_size]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[esm_model.num_layers], return_contacts=False)
    
    # Extract embeddings and remove start/end tokens
    embeddings = results["representations"][esm_model.num_layers][:, 1:-1, :]
    all_embeddings.extend([emb.cpu() for emb in embeddings])

# Pad embeddings to the maximum length
padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)

## 4. Encode Labels and Create Dataset


In [ ]:
def encode_labels(ss_labels, vocab, max_len):
    encoded = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss]
        if len(ids) < max_len:
            ids.extend([-1] * (max_len - len(ids)))
        else:
            ids = ids[:max_len]
        encoded.append(torch.tensor(ids, dtype=torch.long))
    return pad_sequence(encoded, batch_first=True, padding_value=-1)

ss8_labels = encode_labels(df["sst8"], ss8_vocab, max_len)
ss3_labels = encode_labels(df["sst3"], ss3_vocab, max_len)

class ProteinDataset(Dataset):
    def __init__(self, embeddings, sst8_labels, sst3_labels):
        self.embeddings = embeddings
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.sst8_labels[idx], self.sst3_labels[idx]


## 5. K-Fold Cross-Validation Setup
Prepare data for K-fold cross-validation with train/validation splits.

In [ ]:
from sklearn.model_selection import KFold

# First, separate test set (10% of data)
all_indices = np.arange(len(padded_embeddings))
train_val_indices, test_indices = train_test_split(all_indices, test_size=0.1, random_state=42)

# Prepare test dataset
test_dataset = ProteinDataset(padded_embeddings[test_indices], ss8_labels[test_indices], ss3_labels[test_indices])
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Set up 5-fold cross-validation on remaining data
n_folds = 5
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

embedding_dim = padded_embeddings.shape[-1]
print(f"Embedding dimension: {embedding_dim}")
print(f"Training+Validation samples: {len(train_val_indices)}, Test samples: {len(test_indices)}")
print(f"Using {n_folds}-fold cross-validation")

## 6. CNN Model


In [ ]:
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=640, num_filters=128, dropout=0.1):
        super().__init__()
        
        # 1D Convolutional layers
        # nn.Conv1d expects input as (batch, channels, length)
        # Our input is (batch, length, channels), so we'll permute it.
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        self.conv3 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=7, padding=3)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)

        # Two separate classifier heads
        self.q8_head = nn.Linear(num_filters, 8)
        self.q3_head = nn.Linear(num_filters, 3)

    def forward(self, x, mask=None): # Mask is not used by CNN, but kept for compatibility
        """
        x: [batch_size, seq_len, input_dim] (embeddings)
        """
        # Permute from [batch, seq_len, channels] to [batch, channels, seq_len]
        x = x.permute(0, 2, 1)
        
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.dropout2(self.relu2(self.conv2(x)))
        x = self.dropout3(self.relu3(self.conv3(x)))
        
        # Permute back to [batch, seq_len, channels]
        x = x.permute(0, 2, 1)
        
        # Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits

## 7. Training Loop with K-Fold Cross-Validation and Early Stopping
**Training Configuration:**
- **K-Fold Cross-Validation**: 5 folds
- **Epochs**: 50 epochs per fold
- **Early Stopping**: Patience of 5 epochs (stops if validation accuracy doesn't improve)
- **Model**: ProteinCNN with 1D convolutional layers

In [ ]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# K-Fold Cross-Validation
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)

num_epochs = 50
patience = 5
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{n_folds}")
    print(f"{'='*50}")
    
    # Get actual indices for this fold
    fold_train_indices = train_val_indices[train_idx]
    fold_val_indices = train_val_indices[val_idx]
    
    # Create datasets for this fold
    train_dataset = ProteinDataset(padded_embeddings[fold_train_indices], 
                                   ss8_labels[fold_train_indices], 
                                   ss3_labels[fold_train_indices])
    val_dataset = ProteinDataset(padded_embeddings[fold_val_indices], 
                                 ss8_labels[fold_val_indices], 
                                 ss3_labels[fold_val_indices])
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    # Initialize model for this fold
    model = ProteinCNN(input_dim=embedding_dim)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    
    # Optimizer for this fold
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    
    # Early stopping variables
    best_val_acc_q8 = 0.0
    epochs_no_improve = 0
    best_model_state = None
    
    # Training loop for this fold
    for epoch in range(num_epochs):
        model.train()
        train_loss, train_acc_q8, train_acc_q3 = 0, 0, 0
        
        for embeddings, ss8, ss3 in tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}/{num_epochs}"):
            embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
            
            # Forward pass
            q8_logits, q3_logits = model(embeddings, mask=None)
            
            # Loss
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_acc_q8 += compute_accuracy(q8_logits, ss8)
            train_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        train_loss /= len(train_loader)
        train_acc_q8 /= len(train_loader)
        train_acc_q3 /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss, val_acc_q8, val_acc_q3 = 0, 0, 0
        with torch.no_grad():
            for embeddings, ss8, ss3 in val_loader:
                embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
                q8_logits, q3_logits = model(embeddings, mask=None)
                
                loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                loss = loss_q8 + 0.5 * loss_q3
                
                val_loss += loss.item()
                val_acc_q8 += compute_accuracy(q8_logits, ss8)
                val_acc_q3 += compute_accuracy(q3_logits, ss3)
        
        val_loss /= len(val_loader)
        val_acc_q8 /= len(val_loader)
        val_acc_q3 /= len(val_loader)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
        print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
        print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}")
        
        # Early stopping check
        if val_acc_q8 > best_val_acc_q8:
            best_val_acc_q8 = val_acc_q8
            epochs_no_improve = 0
            best_model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            print(f"✓ New best model! Val Acc Q8: {best_val_acc_q8:.4f}")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    # Save best model for this fold
    torch.save(best_model_state, f"best_cnn_model_fold{fold+1}.pt")
    fold_results.append({
        'fold': fold + 1,
        'best_val_acc_q8': best_val_acc_q8,
        'best_val_acc_q3': val_acc_q3
    })
    print(f"\nFold {fold+1} Best Val Acc Q8: {best_val_acc_q8:.4f}")

# Summary of all folds
print(f"\n{'='*50}")
print("K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*50}")
for result in fold_results:
    print(f"Fold {result['fold']}: Val Acc Q8 = {result['best_val_acc_q8']:.4f}")
avg_val_acc = np.mean([r['best_val_acc_q8'] for r in fold_results])
print(f"\nAverage Val Acc Q8 across all folds: {avg_val_acc:.4f}")

# Load best fold model for testing (highest validation accuracy)
best_fold = max(fold_results, key=lambda x: x['best_val_acc_q8'])
print(f"\nUsing model from Fold {best_fold['fold']} for final testing")

## 8. Final Evaluation on Test Set
Evaluates the best model from K-fold cross-validation on the held-out test set.

In [ ]:
# Initialize a new model instance and load the best fold's model
model = ProteinCNN(input_dim=embedding_dim)
model.load_state_dict(torch.load(f"best_cnn_model_fold{best_fold['fold']}.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0, 0, 0
with torch.no_grad():
    for embeddings, ss8, ss3 in test_loader:
        embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
        q8_logits, q3_logits = model(embeddings, mask=None)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"\n{'='*50}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")